# TinyCeNN-LM — Transformer → CeNN Teacher Distillation

This notebook removes Tiny-LLM's only Transformer decoder layer and trains a **CeNN-only replacement core** using a frozen `arnir0/Tiny-LLM` teacher. The objective combines next-token CE, teacher-logit KL distillation, and hidden-state cosine matching.

Default CeNN: 7 shared recurrent steps with dilations `1,2,4,8,16,32,64`, giving a 255-token causal receptive field for a 256-token context.

In [ ]:
# GPU check
import subprocess, sys, pathlib, importlib
subprocess.run(["nvidia-smi"], check=False)


In [ ]:
# Clone/update and install into the active notebook kernel
REPO_DIR = pathlib.Path("/content/TinyCeNN-LM")
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
importlib.invalidate_caches()
import tinycenn_lm
print("TinyCeNN-LM:", tinycenn_lm.__file__)


## Hugging Face login
Add a Hugging Face **write** token to Colab Secrets as `HF_TOKEN`. Do not paste a real token into this public notebook.

In [ ]:
from huggingface_hub import login, HfApi
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    login()
api = HfApi()
hf_user = api.whoami()["name"]
print("Logged in as:", hf_user)


In [ ]:
# Distillation configuration
MAX_TOKENS = 10_000_000
CONTEXT_LENGTH = 256
BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 1e-3
CENN_STEPS = 7
DILATIONS = "1,2,4,8,16,32,64"
TEMPERATURE = 2.0
CE_WEIGHT = 1.0
KL_WEIGHT = 1.0
HIDDEN_WEIGHT = 0.25
OUTPUT_DIR = str(REPO_DIR / "checkpoints/cenn-student-distill")


In [ ]:
# Train Transformer-free CeNN student from the frozen Tiny-LLM teacher
cmd = [
    sys.executable, str(REPO_DIR / "scripts/train_distill.py"),
    "--max-tokens", str(MAX_TOKENS),
    "--context-length", str(CONTEXT_LENGTH),
    "--batch-size", str(BATCH_SIZE),
    "--grad-accum", str(GRAD_ACCUM),
    "--learning-rate", str(LEARNING_RATE),
    "--steps", str(CENN_STEPS),
    "--dilations", DILATIONS,
    "--temperature", str(TEMPERATURE),
    "--ce-weight", str(CE_WEIGHT),
    "--kl-weight", str(KL_WEIGHT),
    "--hidden-weight", str(HIDDEN_WEIGHT),
    "--output-dir", OUTPUT_DIR,
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=str(REPO_DIR), check=True)


In [ ]:
# Inspect distillation health
import json
report_path = pathlib.Path(OUTPUT_DIR) / "distillation_report.json"
report = json.loads(report_path.read_text())
print(json.dumps(report, indent=2))
if report["status"] == "diverged":
    raise RuntimeError("Distillation diverged; checkpoint will not be published.")
print("Teacher gap recovery:", f"{100*report['teacher_gap_recovery_fraction']:.2f}%")


In [ ]:
# Select best checkpoint and create a model card
best_dir = pathlib.Path(OUTPUT_DIR + "-best")
publish_dir = best_dir if best_dir.exists() else pathlib.Path(OUTPUT_DIR)
print("Publishing:", publish_dir)
card = f'''---
base_model: arnir0/Tiny-LLM
library_name: transformers
pipeline_tag: text-generation
tags:
- cenn
- knowledge-distillation
- transformer-free
- language-modeling
- recurrent-neural-network
---

# TinyCeNN-LM Distilled

Transformer-free CeNN student distilled from `arnir0/Tiny-LLM`.

- Transformer layers remaining: 0
- CeNN recurrent steps: {report['cenn_steps']}
- CeNN receptive field: {report['cenn_receptive_field']} tokens
- Training tokens: {report['seen_tokens']:,}
- Best student CE: {report['best']['student_ce']:.6f}
- Teacher CE: {report['best']['teacher_ce']:.6f}
- Teacher-gap recovery: {100*report['teacher_gap_recovery_fraction']:.2f}%

Load with the `TinyCeNN-LM` GitHub package via `build_cenn_student()`.
'''
(publish_dir / "README.md").write_text(card, encoding="utf-8")
if report_path.exists() and publish_dir != report_path.parent:
    (publish_dir / "distillation_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")


In [ ]:
# Upload the distilled student to Hugging Face
HF_MODEL_NAME = "TinyCeNN-LM-Distilled"
HF_REPO_ID = f"{hf_user}/{HF_MODEL_NAME}"
api.create_repo(HF_REPO_ID, repo_type="model", private=False, exist_ok=True)
api.upload_folder(
    repo_id=HF_REPO_ID, repo_type="model", folder_path=str(publish_dir),
    commit_message=f"Upload CeNN-only distilled student ({report['seen_tokens']:,} tokens)",
)
print(f"https://huggingface.co/{HF_REPO_ID}")


In [ ]:
# Download the published artifact back from Hugging Face
from huggingface_hub import snapshot_download
downloaded_student = snapshot_download(repo_id=HF_REPO_ID, repo_type="model")
print(downloaded_student)


In [ ]:
# Rebuild the Transformer-free student and run generation tests
import torch
from transformers import AutoTokenizer
from tinycenn_lm import build_cenn_student, CeNNReplacementLayer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else (torch.float16 if device.type == "cuda" else torch.float32)
student = build_cenn_student(downloaded_student, device=device, dtype=dtype)
student.eval()
tokenizer = AutoTokenizer.from_pretrained(downloaded_student)
replacement_layers = [m for m in student.modules() if isinstance(m, CeNNReplacementLayer)]
assert len(replacement_layers) == 1
assert not any("self_attn" in name or "mlp" in name for name, _ in student.named_modules())
print("Transformer-free structure: PASS")
prompts = [
    "The capital of Austria is",
    "Artificial intelligence can help",
    "A small language model",
    "In the future, efficient AI",
]
for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.inference_mode():
        out = student.generate(**inputs, max_new_tokens=40, do_sample=False, use_cache=False, pad_token_id=tokenizer.eos_token_id)
    print("\nPROMPT:", prompt)
    print(tokenizer.decode(out[0], skip_special_tokens=True))


In [ ]:
# Numerical sanity test after remote reload
import math
sample = tokenizer("TinyCeNN is a recurrent language model.", return_tensors="pt").to(device)
with torch.inference_mode():
    outputs = student(**sample, labels=sample["input_ids"], use_cache=False)
loss = float(outputs.loss.detach().float().cpu())
print("sanity loss:", loss, "ppl:", math.exp(min(loss, 20)))
assert math.isfinite(loss)
print("Remote reload + CeNN-only inference: PASS")
